In [37]:
import sys
import os
from pathlib import Path

# Get the absolute path to the project root
project_root = Path.cwd().parent

# Add the project root to Python's import path
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Can now import files as if the notebook was located in the root folder

# Introduction
Notebook for testing that the parquet storage class works together with the yfinance connector

## Step 1: Check that imports work

In [44]:
# Import relevant files
from src.config import settings
from src.storage.parquet_storage import ParquetStorage
from src.data.connector_factory import ConnectorFactory

# For testing parquet storage
settings.STORAGE_BACKEND = "parquet"

## Step 2: Check that the folder is created correctly and that the storage used in yfinance is correct

In [39]:
# Create the storage backend so we can inspect it directly
storage = ParquetStorage()
print(f"Cache directory: {storage.cache_dir}")
print(f"Does cache dir exist? {os.path.exists(storage.cache_dir)}")

# Create the connector with the storage injected
connector = ConnectorFactory.get_connector(symbol="SPY")
print(f"Connector: {connector.__class__.__name__}")
print(f"Storage: {connector.storage.__class__.__name__}")

Cache directory: C:\Users\Danny\Desktop\Stuff\Projects\Projects - Finance\Market Making Bot\Volatility-Adaptive-Options-Market-Making-Engine\data\cache
Does cache dir exist? True
Connector: YFinanceConnector
Storage: ParquetStorage


## Step 3: Fetch Data

In [46]:
import time

expiries = connector.get_available_expiries()
print(f"experies = {expiries}")

if not expiries:
    print("No expiries found. Check yfinance connection.")
else:
    first_expiry = expiries[0]
    print(f"Testing with expiry: {first_expiry}")

    # Time the fetch
    start = time.time()
    data = connector.get_chain_for_expiry(first_expiry, use_cache=False)
    elapsed = time.time() - start

    calls = data.get("calls", [])
    puts = data.get("puts", [])

    print(f"Fetched: {len(calls)} calls, {len(puts)} puts")
    print(f"Time taken: {elapsed:.4f} seconds")
    print(f"A parquet file under data/cache should be created with the name: SPY_{first_expiry}.parquet")

experies = ('2026-07-06', '2026-07-07', '2026-07-08', '2026-07-09', '2026-07-10', '2026-07-17', '2026-07-24', '2026-07-31', '2026-08-07', '2026-08-21', '2026-08-31', '2026-09-18', '2026-09-30', '2026-10-16', '2026-10-30', '2026-11-20', '2026-11-30', '2026-12-18', '2026-12-31', '2027-01-15', '2027-03-19', '2027-03-31', '2027-06-17', '2027-09-17', '2027-12-17', '2028-01-21', '2028-06-16', '2028-12-15')
Testing with expiry: 2026-07-06
Fetched: 54 calls, 52 puts
Time taken: 0.2735 seconds
A parquet file under data/cache should be created with the name: SPY_2026-07-06.parquet


## Step 4: Use cached data when fetching - Should be faster

In [41]:
start = time.time()
cached_data = connector.get_chain_for_expiry(first_expiry, use_cache=True)
elapsed = time.time() - start

cached_calls = cached_data.get("calls", [])
cached_puts = cached_data.get("puts", [])

print(f"Loaded from cache: {len(cached_calls)} calls, {len(cached_puts)} puts")
print(f"Time taken (cache): {elapsed:.4f} seconds")

# Inspect saved data
print(f"Cached Calls: {cached_calls}")
print(f"Cached Puts: {cached_puts}")

Loaded from cache: 54 calls, 52 puts
Time taken (cache): 0.0235 seconds
Cached Calls: [OptionQuote(strike=625.0, bid=117.82, ask=121.34, mid=119.58, implied_vol=1.023930661621094, volume=30, open_interest=31, option_type='call'), OptionQuote(strike=660.0, bid=82.83, ask=86.35, mid=84.59, implied_vol=0.7605004418945314, volume=6, open_interest=26, option_type='call'), OptionQuote(strike=665.0, bid=77.83, ask=81.35, mid=79.59, implied_vol=0.7229031616210937, volume=2, open_interest=0, option_type='call'), OptionQuote(strike=670.0, bid=72.83, ask=76.35, mid=74.59, implied_vol=0.6850617431640625, volume=2, open_interest=12, option_type='call'), OptionQuote(strike=700.0, bid=42.84, ask=46.36, mid=44.6, implied_vol=0.45654840332031255, volume=122, open_interest=5, option_type='call'), OptionQuote(strike=703.0, bid=39.84, ask=43.36, mid=41.6, implied_vol=0.4331111376953125, volume=8, open_interest=8, option_type='call'), OptionQuote(strike=705.0, bid=37.85, ask=41.36, mid=39.605000000000004, 

# Plot data

In [ ]:
import matplotlib.pyplot as plt

